# 06 — Scientific evaluation

**Purpose:** Validate prediction coverage, source hashes and label set before reporting macro-F1, accuracy and invalid-output rates.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


This notebook is deliberately strict. It rejects predictions for the wrong task, partial samples, wrong dataset versions, unknown example IDs and duplicate IDs. Invalid generated labels are counted as mistakes, not silently discarded.

In [ ]:
import pandas as pd
TASK='sib200'; LANG='ur'; SPLIT='validation'  # switch to 'test' only at final evaluation
paths=sorted((ROOT/'outputs/predictions').glob(f'*_{LANG}_{SPLIT}.csv'))
print('Found',len(paths),'prediction files')
for p in paths: print(p.name)

In [ ]:
results=[]
rows=load_task(TASK,LANG,SPLIT); labels=task_labels(TASK)
for p in paths:
    meta_path=p.with_suffix('.meta.json')
    if not meta_path.exists(): print('SKIP (no metadata):',p); continue
    meta=read_json(meta_path)
    if meta['task']!=TASK: print('SKIP other task:',p.name); continue
    try:
        validate_prediction_metadata(meta,TASK,LANG,SPLIT)
        if meta.get('subset'): raise ValueError('partial/smoke prediction set cannot be called full evaluation')
        predictions=read_csv(p)
        score=evaluate_predictions(rows,predictions,labels)
        compact={k:v for k,v in score.items() if k not in ['classification_report','confusion_matrix','confusion_labels']}
        compact.update({'run_id':meta['run_id'],'method':meta.get('method',meta['run_id']),
                        'task':TASK,'lang':LANG,'split':SPLIT,'file':str(p)})
        results.append(compact)
        write_json(ROOT/'outputs/metrics'/f'{p.stem}_metrics.json',dict(**compact, details=score, metadata=meta))
        print(p.name,'accuracy',round(score['accuracy'],4),'macro-F1',round(score['macro_f1'],4),
              'invalid',score['invalid_count'])
    except (ValueError,KeyError,FileNotFoundError) as e:
        print('REJECTED',p.name,'reason:',e)
if results:
    df=pd.DataFrame(results).sort_values('macro_f1',ascending=False)
    display(df[['run_id','split','count','accuracy','macro_f1','invalid_count']])
    df.to_csv(ROOT/'outputs/metrics'/f'{TASK}_{LANG}_{SPLIT}_leaderboard.csv',index=False)
else: print('No complete, verified predictions available yet.')